In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path

# check if workding_dir is in local variables
if "workding_dir" not in locals():
    workding_dir = str(Path.cwd().parent)

os.chdir(workding_dir)
sys.path.append(workding_dir)
print("workding dir:", workding_dir)

workding dir: /Users/inflaton/code/engd/papers/multi-agent-llm-at-edge


In [3]:
from dotenv import find_dotenv, load_dotenv

found_dotenv = find_dotenv(".env")

if len(found_dotenv) == 0:
    found_dotenv = find_dotenv(".env.example")
print(f"loading env vars from: {found_dotenv}")
load_dotenv(found_dotenv, override=True)

loading env vars from: /Users/inflaton/code/engd/papers/multi-agent-llm-at-edge/.env


True

In [4]:
from src.misc.metrics import *

In [5]:
import pandas as pd

df_emails = pd.read_csv("dataset/emails.csv")
df_emails = df_emails[:990]
df_emails.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   email_id             990 non-null    object 
 1   sender_email         990 non-null    object 
 2   recipient_email      990 non-null    object 
 3   subject              990 non-null    object 
 4   email_body           990 non-null    object 
 5   attachments          990 non-null    object 
 6   process_status       990 non-null    object 
 7   response             0 non-null      float64
 8   start_time           0 non-null      float64
 9   end_time             0 non-null      float64
 10  full_logs            0 non-null      float64
 11  total_time           990 non-null    int64  
 12  successful_requests  990 non-null    int64  
 13  total_tokens         990 non-null    int64  
 14  prompt_tokens        990 non-null    int64  
 15  completion_tokens    990 non-null    int

In [6]:
df_transactions = pd.read_csv("dataset/transactions.csv")
df_transactions = df_transactions[:990]
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   invoice_id            990 non-null    int64  
 1   bank_name             990 non-null    object 
 2   transaction_id        990 non-null    object 
 3   amount                990 non-null    float64
 4   recipient_name        990 non-null    object 
 5   sender_name           990 non-null    object 
 6   reconciliation_state  990 non-null    object 
 7   email_details         0 non-null      float64
dtypes: float64(2), int64(1), object(5)
memory usage: 62.0+ KB


In [7]:
df_ground_truth = pd.read_csv("dataset/ground_truth.csv")
df_ground_truth.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   invoice_id           1980 non-null   int64  
 1   bank_name            1980 non-null   object 
 2   transaction_id       1980 non-null   object 
 3   transaction_date     1980 non-null   object 
 4   amount               1980 non-null   float64
 5   recipient_name       1980 non-null   object 
 6   sender_name          1980 non-null   object 
 7   email_id             1980 non-null   object 
 8   sender_email         1980 non-null   object 
 9   recipient_email      1980 non-null   object 
 10  subject              1980 non-null   object 
 11  email_body           1980 non-null   object 
 12  filename             990 non-null    object 
 13  expected_ocr_result  1980 non-null   object 
dtypes: float64(1), int64(1), object(12)
memory usage: 216.7+ KB


In [8]:
df_ground_truth.iloc[0]

invoice_id                                                         43925
bank_name                                                       Bank 538
transaction_id                                          EC4F9FE854344D25
transaction_date                                                 4/23/23
amount                                                             196.0
recipient_name                                                     Tanya
sender_name                                                 Robin Levine
email_id                                93185A89130149C0A842968E4AFDCAA2
sender_email                                     RobinLevine@example.com
recipient_email                             tanya.official.456@gmail.com
subject                       Payment Confirmation for Invoice ID: 43925
email_body             Hi Tanya ! Please find attached payment screen...
filename                                         imgs/transaction_1.jpeg
expected_ocr_result    Bank 538\nSent by: Robin Lev

In [9]:
def get_ocr_result(row):
    return (
        f"""{row['bank_name']}
Sent by: {row['sender_name']}
You paid {row['amount']} to {row['recipient_name']}
Transaction Reference Number: {row['transaction_id']}
Transaction Date: {row['transaction_date']}
Invoice ID: {row['invoice_id']}
"""
        if isinstance(row["filename"], str)
        else None
    )


df_ground_truth["expected_ocr_result"] = df_ground_truth.apply(get_ocr_result, axis=1)
df_ground_truth.iloc[0]

invoice_id                                                         43925
bank_name                                                       Bank 538
transaction_id                                          EC4F9FE854344D25
transaction_date                                                 4/23/23
amount                                                             196.0
recipient_name                                                     Tanya
sender_name                                                 Robin Levine
email_id                                93185A89130149C0A842968E4AFDCAA2
sender_email                                     RobinLevine@example.com
recipient_email                             tanya.official.456@gmail.com
subject                       Payment Confirmation for Invoice ID: 43925
email_body             Hi Tanya ! Please find attached payment screen...
filename                                         imgs/transaction_1.jpeg
expected_ocr_result    Bank 538\nSent by: Robin Lev

In [10]:
df_ground_truth.iloc[-1]

invoice_id                                                       1042392
bank_name                                                       Bank 590
transaction_id                                        T-418BBE70721C46FD
transaction_date                                                12/20/24
amount                                                             147.0
recipient_name                                                     Tanya
sender_name                                                    Ruth West
email_id                              E-246AB042D6C74C53B38B68BDBDA07590
sender_email                                        RuthWest@example.com
recipient_email                             tanya.official.456@gmail.com
subject                     Payment Confirmation for Invoice ID: 1042392
email_body             Dear Tanya, I hope this message finds you well...
filename                                                             NaN
expected_ocr_result                                

In [11]:
df_ground_truth.to_csv("dataset/ground_truth.csv", index=False)

In [12]:
df_ground_truth = df_ground_truth[:990]
df_ground_truth.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   invoice_id           990 non-null    int64  
 1   bank_name            990 non-null    object 
 2   transaction_id       990 non-null    object 
 3   transaction_date     990 non-null    object 
 4   amount               990 non-null    float64
 5   recipient_name       990 non-null    object 
 6   sender_name          990 non-null    object 
 7   email_id             990 non-null    object 
 8   sender_email         990 non-null    object 
 9   recipient_email      990 non-null    object 
 10  subject              990 non-null    object 
 11  email_body           990 non-null    object 
 12  filename             990 non-null    object 
 13  expected_ocr_result  990 non-null    object 
dtypes: float64(1), int64(1), object(12)
memory usage: 108.4+ KB


In [13]:
email_subject = """Payment Confirmation for Invoice ID: {invoice_id}"""
email_body = """Dear {recipient_name}, I hope this message finds you well.I am writing to confirm that we have successfully made the payment for Invoice ID: {invoice_id}, related to our recent order with you. Here are the details:  
Invoice ID: {invoice_id}
Invoice Date: {transaction_date}
Total Amount paid: {amount}
Transaction Reference Number - {transaction_id}
Thank you and I look forward to our continued business
Best regards,
{sender_name}"""

In [14]:
def transform(row):
    row["invoice_id"] = 1000000 + row["invoice_id"]
    row["transaction_id"] = "T-" + row["transaction_id"]
    row["email_id"] = "E-" + row["email_id"]

    invoice_id = row["invoice_id"]
    recipient_name = row["recipient_name"]
    transaction_date = row["transaction_date"]
    amount = row["amount"]
    transaction_id = row["transaction_id"]
    sender_name = row["sender_name"]

    subject = email_subject.format(invoice_id=invoice_id)
    body = email_body.format(
        recipient_name=recipient_name,
        invoice_id=invoice_id,
        transaction_date=transaction_date,
        amount=amount,
        transaction_id=transaction_id,
        sender_name=sender_name,
    )

    row["subject"] = subject
    row["email_body"] = body
    row["filename"] = None
    row["expected_ocr_result"] = None
    return row

In [15]:
df_ground_truth_no_image = df_ground_truth.apply(transform, axis=1)
df_ground_truth_no_image.head()

,invoice_id,bank_name,transaction_id,transaction_date,amount,recipient_name,sender_name,email_id,sender_email,recipient_email,subject,email_body,filename,expected_ocr_result
0,1043925,Bank 538,T-EC4F9FE854344D25,4/23/23,196.0,Tanya,Robin Levine,E-93185A89130149C0A842968E4AFDCAA2,RobinLevine@example.com,tanya.official.456@gmail.com,Payment Confirmation for Invoice ID: 1043925,"Dear Tanya, I hope this message finds you well...",None,None
1,1051782,Bank 533,T-3D2ADBBC17B641C4,6/5/24,159.0,Tanya,Steven Nixon,E-614972488B7B411BAEA4814CF066CDAA,StevenNixon@example.com,tanya.official.456@gmail.com,Payment Confirmation for Invoice ID: 1051782,"Dear Tanya, I hope this message finds you well...",None,None
2,1042968,Bank 972,T-C516A7A9C9534B6A,5/3/23,21.0,Tanya,Madison Ford,E-C484ABEB38F44D29960B1CEF58510627,MadisonFord@example.com,tanya.official.456@gmail.com,Payment Confirmation for Invoice ID: 1042968,"Dear Tanya, I hope this message finds you well...",None,None
3,1092948,Bank 337,T-43532033DC0E44D6,5/2/24,94.0,Tanya,Kathryn Jones,E-706B3403E8024178948B3BFBF9E64A25,KathrynJones@example.com,tanya.official.456@gmail.com,Payment Confirmation for Invoice ID: 1092948,"Dear Tanya, I hope this message finds you well...",None,None
4,1020304,Bank 924,T-7DAA6E4202B24892,2/28/23,48.0,Tanya,Christine Owens,E-86838440CA6C472AACFA3FCE1382609C,ChristineOwens@example.com,tanya.official.456@gmail.com,Payment Confirmation for Invoice ID: 1020304,"Dear Tanya, I hope this message finds you well...",None,None


In [16]:
row = df_ground_truth_no_image.iloc[0]
# print all elements in the row

for col in df_ground_truth_no_image.columns:
    print(f"{col}: {row[col]}")

invoice_id: 1043925
bank_name: Bank 538
transaction_id: T-EC4F9FE854344D25
transaction_date: 4/23/23
amount: 196.0
recipient_name: Tanya
sender_name: Robin Levine
email_id: E-93185A89130149C0A842968E4AFDCAA2
sender_email: RobinLevine@example.com
recipient_email: tanya.official.456@gmail.com
subject: Payment Confirmation for Invoice ID: 1043925
email_body: Dear Tanya, I hope this message finds you well.I am writing to confirm that we have successfully made the payment for Invoice ID: 1043925, related to our recent order with you. Here are the details:  
Invoice ID: 1043925
Invoice Date: 4/23/23
Total Amount paid: 196.0
Transaction Reference Number - T-EC4F9FE854344D25
Thank you and I look forward to our continued business
Best regards,
Robin Levine
filename: None
expected_ocr_result: None


In [17]:
row = df_ground_truth_no_image.iloc[-1]
# print all elements in the row

for col in df_ground_truth_no_image.columns:
    print(f"{col}: {row[col]}")

invoice_id: 1042392
bank_name: Bank 590
transaction_id: T-418BBE70721C46FD
transaction_date: 12/20/24
amount: 147.0
recipient_name: Tanya
sender_name: Ruth West
email_id: E-246AB042D6C74C53B38B68BDBDA07590
sender_email: RuthWest@example.com
recipient_email: tanya.official.456@gmail.com
subject: Payment Confirmation for Invoice ID: 1042392
email_body: Dear Tanya, I hope this message finds you well.I am writing to confirm that we have successfully made the payment for Invoice ID: 1042392, related to our recent order with you. Here are the details:  
Invoice ID: 1042392
Invoice Date: 12/20/24
Total Amount paid: 147.0
Transaction Reference Number - T-418BBE70721C46FD
Thank you and I look forward to our continued business
Best regards,
Ruth West
filename: None
expected_ocr_result: None


In [18]:
df_emails_no_image = df_emails.copy()
# set attachments to None
df_emails_no_image["attachments"] = None
df_emails_no_image["email_id"] = df_ground_truth_no_image["email_id"]
df_emails_no_image["subject"] = df_ground_truth_no_image["subject"]
df_emails_no_image["email_body"] = df_ground_truth_no_image["email_body"]

In [19]:
row = df_emails_no_image.iloc[0]
# print all elements in the row

for col in df_emails_no_image.columns:
    print(f"{col}: {row[col]}")

email_id: E-93185A89130149C0A842968E4AFDCAA2
sender_email: RobinLevine@example.com
recipient_email: tanya.official.456@gmail.com
subject: Payment Confirmation for Invoice ID: 1043925
email_body: Dear Tanya, I hope this message finds you well.I am writing to confirm that we have successfully made the payment for Invoice ID: 1043925, related to our recent order with you. Here are the details:  
Invoice ID: 1043925
Invoice Date: 4/23/23
Total Amount paid: 196.0
Transaction Reference Number - T-EC4F9FE854344D25
Thank you and I look forward to our continued business
Best regards,
Robin Levine
attachments: None
process_status: NOT_STARTED
response: nan
start_time: nan
end_time: nan
full_logs: nan
total_time: 0
successful_requests: 0
total_tokens: 0
prompt_tokens: 0
completion_tokens: 0
total_cost: 0


In [20]:
df_transactions_no_image = df_transactions.copy()
df_transactions_no_image["invoice_id"] = df_ground_truth_no_image["invoice_id"]
df_transactions_no_image["transaction_id"] = df_ground_truth_no_image["transaction_id"]

In [21]:
row = df_transactions_no_image.iloc[0]
# print all elements in the row
for col in df_transactions_no_image.columns:
    print(f"{col}: {row[col]}")

invoice_id: 1043925
bank_name: Bank 538
transaction_id: T-EC4F9FE854344D25
amount: 196.0
recipient_name: Tanya
sender_name: Robin Levine
reconciliation_state: UNPAID
email_details: nan


In [22]:
df_emails = pd.concat([df_emails, df_emails_no_image], ignore_index=True)
df_emails.to_csv("dataset/emails.csv", index=False)
df_emails = pd.read_csv("dataset/emails.csv")
df_emails.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   email_id             1980 non-null   object 
 1   sender_email         1980 non-null   object 
 2   recipient_email      1980 non-null   object 
 3   subject              1980 non-null   object 
 4   email_body           1980 non-null   object 
 5   attachments          990 non-null    object 
 6   process_status       1980 non-null   object 
 7   response             0 non-null      float64
 8   start_time           0 non-null      float64
 9   end_time             0 non-null      float64
 10  full_logs            0 non-null      float64
 11  total_time           1980 non-null   int64  
 12  successful_requests  1980 non-null   int64  
 13  total_tokens         1980 non-null   int64  
 14  prompt_tokens        1980 non-null   int64  
 15  completion_tokens    1980 non-null   i

In [23]:
df_transactions = pd.concat(
    [df_transactions, df_transactions_no_image], ignore_index=True
)
df_transactions.to_csv("dataset/transactions.csv", index=False)
df_transactions = pd.read_csv("dataset/transactions.csv")
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   invoice_id            1980 non-null   int64  
 1   bank_name             1980 non-null   object 
 2   transaction_id        1980 non-null   object 
 3   amount                1980 non-null   float64
 4   recipient_name        1980 non-null   object 
 5   sender_name           1980 non-null   object 
 6   reconciliation_state  1980 non-null   object 
 7   email_details         0 non-null      float64
dtypes: float64(2), int64(1), object(5)
memory usage: 123.9+ KB


In [24]:
df_ground_truth = pd.concat(
    [df_ground_truth, df_ground_truth_no_image], ignore_index=True
)
df_ground_truth.to_csv("dataset/ground_truth.csv", index=False)
df_ground_truth = pd.read_csv("dataset/ground_truth.csv")
df_ground_truth.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   invoice_id           1980 non-null   int64  
 1   bank_name            1980 non-null   object 
 2   transaction_id       1980 non-null   object 
 3   transaction_date     1980 non-null   object 
 4   amount               1980 non-null   float64
 5   recipient_name       1980 non-null   object 
 6   sender_name          1980 non-null   object 
 7   email_id             1980 non-null   object 
 8   sender_email         1980 non-null   object 
 9   recipient_email      1980 non-null   object 
 10  subject              1980 non-null   object 
 11  email_body           1980 non-null   object 
 12  filename             990 non-null    object 
 13  expected_ocr_result  990 non-null    object 
dtypes: float64(1), int64(1), object(12)
memory usage: 216.7+ KB


In [28]:
print_row_details(df_ground_truth, [0])

--------------------------------------------------
invoice_id: 43925
--------------------------------------------------
bank_name: Bank 538
--------------------------------------------------
transaction_id: EC4F9FE854344D25
--------------------------------------------------
transaction_date: 4/23/23
--------------------------------------------------
amount: 196.0
--------------------------------------------------
recipient_name: Tanya
--------------------------------------------------
sender_name: Robin Levine
--------------------------------------------------
email_id: 93185A89130149C0A842968E4AFDCAA2
--------------------------------------------------
sender_email: RobinLevine@example.com
--------------------------------------------------
recipient_email: tanya.official.456@gmail.com
--------------------------------------------------
subject: Payment Confirmation for Invoice ID: 43925
--------------------------------------------------
email_body: Hi Tanya ! Please find attached paymen

In [29]:
print_row_details(df_ground_truth, [-1])

--------------------------------------------------
invoice_id: 1042392
--------------------------------------------------
bank_name: Bank 590
--------------------------------------------------
transaction_id: T-418BBE70721C46FD
--------------------------------------------------
transaction_date: 12/20/24
--------------------------------------------------
amount: 147.0
--------------------------------------------------
recipient_name: Tanya
--------------------------------------------------
sender_name: Ruth West
--------------------------------------------------
email_id: E-246AB042D6C74C53B38B68BDBDA07590
--------------------------------------------------
sender_email: RuthWest@example.com
--------------------------------------------------
recipient_email: tanya.official.456@gmail.com
--------------------------------------------------
subject: Payment Confirmation for Invoice ID: 1042392
--------------------------------------------------
email_body: Dear Tanya, I hope this message fin